# DAAAnalyzer_2.0.AnalyzeABRs.ipynb
Created by: JFranco | Created On: 27 NOV 2024 | Last Env: QISHA | Last Run Date: 27 NOV 2024

This Python notebook will read in all DPOAE files for a specific DAA Run as specified by the user. It will make a lineplot for all availble files and label them by category. 

In [2]:
import numpy as np
import seaborn as sns
import pandas as pd
import glob
import os
import xlrd
import math
import matplotlib.pyplot as plt

In [4]:
#                *** WHAT TO ANALYZE // WHERE TO GET/STORE **
# Key identifiers
prepID = 'CMI_003'
daaRun = '1'
daaType = 'Pre-Noise Exposure'
isoFile = 'IsoDP-'+daaRun+'-1.tsv'
fnMD = 'CMI_003.Metadata.AnimalInfo.csv'

# Directories 
#   existing ones 
dirMain = '/Users/joyfranco/Dropbox (Partners HealthCare)/JF_Shared/Data/CFTS/'
dirPrep = dirMain+prepID+'/'
dirData = dirPrep+'RawData/'
dirMD = dirPrep+'Metadata/'

#   ones that need to be made    
dirSV = dirPrep+'Results/'        

In [5]:
#    *** LOAD MD SHEETS **
# Sample metadata sheet that include animal and condition info
dfMD = pd.read_csv(dirMD+fnMD)
dfMD.reset_index(inplace=True)
dfMD

,index,AnimalID,Sex,DOB,CMIID,InjectionDate,AgeAtInjection,Package,InjGroup,NE,NEDate,AgeAtNE,NELevel,NEBand,NEDuration,NEGroup
0,0,C016,M,9/24/24,CMI_003,11/11/24,7,AAV,HABclw,Yes,11/25/24,9,97.5,8_16,2h,1
1,1,C017,F,9/24/24,CMI_003,11/11/24,7,AAV,HABclw,Yes,11/25/24,9,97.5,8_16,2h,2
2,2,C018,M,9/24/24,CMI_003,11/11/24,7,AAV,HABclw,Yes,11/25/24,9,97.5,8_16,2h,2
3,3,C019,F,9/24/24,CMI_003,11/11/24,7,AAV,CtrlInj,Yes,11/25/24,9,97.5,8_16,2h,1
4,4,C020,M,9/24/24,CMI_003,11/11/24,7,AAV,CtrlInj,Yes,11/25/24,9,97.5,8_16,2h,1
5,5,C021,F,9/24/24,CMI_003,11/11/24,7,AAV,CtrlInj,Yes,11/25/24,9,97.5,8_16,2h,2
6,6,C022,M,9/24/24,CMI_003,11/11/24,7,AAV,UnInj,Yes,11/25/24,9,97.5,8_16,2h,2
7,7,C023,F,9/24/24,CMI_003,11/11/24,7,AAV,UnInj,Yes,11/25/24,9,97.5,8_16,2h,1


In [6]:
#           *** INITIALIZE RUN SPECIFIC DIRECTORY ETC FOR STORING RESULTS **
# Create directory for storing spreadsheetS and summary plotS for this run
if not os.path.exists(dirSV): os.mkdir(dirSV)

In [7]:
#           *** GENERATE A LIST OF DAA FOLDERS THAT WILL BE ANALYZED **
# Get a list of all the folders that match the prep
allData = os.listdir(dirData)
dataList = []
for datum in allData:
    if ('.'+daaRun) in datum:
        dataList.append(datum)
dataList

['CMI_003.C020.1',
 'CMI_003.C023.1',
 'CMI_003.C016.1',
 'CMI_003.C018.1',
 'CMI_003.C022.1',
 'CMI_003.C017.1',
 'CMI_003.C019.1',
 'CMI_003.C021.1']

In [20]:
#           *** ITERATE THROUGH FOLDERS AND GENERATE ABR PLOTS **
# 1. Open the folder and get all available ABR files
for daa in dataList:
    # Setup the path to the file
    abrFold = dirData+daa+'/'
    # Get a list of available ABR files
    allFiles = os.listdir(abrFold)
    files = []
    for file in allFiles:
        if ("ABR-"+daaRun) in file:
            files.append(file)
    # Go through the files and make a line plot that contains all ABRs for that frequency
    for file in files:
        # Setup the path to the file
        path = dirData+daa+'/'+file
        # Read in the file & make adjustments for readability
        dfABR = pd.read_csv(path, sep='\t', encoding='unicode_escape', lineterminator='\n')
        # Have to get out:
        freq = dfABR.columns.tolist()[6][8:-1]
        # Levels
        # Number of averages -- for each column showing 

In [48]:
str = dfABR.iloc[0].values[0][7:-1]

In [49]:
tries = [i.strip() for i in str.split(';')]
tries

['554', '566', '558', '552', '250']

In [52]:
# This one is going to be harder because the levels are embedded and it'll be hard to say how the indexing changes.
# Should pln on just searching for the word LEVELS: and then get all of the characters until the next "\r"
dfABR.iloc[1].values[0]

array(['\r:CFTS Version: V2.37.2 (R3282)\r:LEVELS:20;25;30;35;40;\r:DATA\r -0.050935',
       0.074373, -0.064534, 0.03543, -0.020103, nan, nan, nan, nan, nan,
       nan, nan], dtype=object)